# VLA Setup & Verification

Verify Kaggle T4 environment: GPU, SmolVLA, LIBERO, dataset, VRAM budget.

In [ ]:
# Cell 1: Install (run once per Kaggle session)
!pip install -q lerobot[smolvla,peft,libero]

# MuJoCo headless rendering
import os
os.environ["MUJOCO_GL"] = "egl"

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
assert torch.cuda.is_available(), "No GPU found!"

In [ ]:
# Cell 2: Verify SmolVLA model loads
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, SmolVLAConfig

config = SmolVLAConfig()
print(f"SmolVLA config loaded: {config.pretrained_path}")

# Quick memory check — load model briefly
policy = SmolVLAPolicy(config, dataset_stats=None)
param_count = sum(p.numel() for p in policy.parameters())
print(f"SmolVLA parameters: {param_count / 1e6:.1f}M")
del policy
torch.cuda.empty_cache()
print("SmolVLA load: OK")

In [ ]:
# Cell 3: Verify LIBERO simulation
import gymnasium as gym
from lerobot.envs.libero import LiberoEnv  # adjust import as needed

# Check LIBERO tasks are available
print("LIBERO environment: OK")
print("MuJoCo rendering backend:", os.environ.get("MUJOCO_GL", "not set"))

In [ ]:
# Cell 4: Verify dataset access
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset("HuggingFaceVLA/libero")
print(f"Dataset loaded: {len(dataset)} frames")
sample = dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Action shape: {sample['action'].shape}")
print(f"State shape: {sample['observation.state'].shape}")
print("Dataset: OK")

In [ ]:
# Cell 5: LoRA training dry run (1 step) to measure VRAM
import subprocess

# Run 1 training step to measure peak memory
result = subprocess.run([
    "lerobot-train",
    "--policy.path=lerobot/smolvla_base",
    "--dataset.repo_id=HuggingFaceVLA/libero",
    "--batch_size=8",
    "--steps=1",
    "--peft.method_type=LORA",
    "--peft.r=32",
    "--wandb.enable=false",
    "--env.type=libero",
    "--env.task=libero_object",
], capture_output=True, text=True, timeout=300)

print("STDOUT:", result.stdout[-500:] if result.stdout else "empty")
print("STDERR:", result.stderr[-500:] if result.stderr else "empty")

# Check peak VRAM
peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM: {peak_mem:.2f} GB / 16 GB")
assert peak_mem < 15.0, f"VRAM too high: {peak_mem:.2f} GB"
print("VRAM budget: OK")